# Spark + Unity Catalog + MinIO Demo

This notebook demonstrates the full flow:
1. Create a Spark session with Unity Catalog connector
2. Create catalog, schema, and table (metadata in UC, data in MinIO)
3. Insert and query data

```
JupyterHub Notebook
      │
      ▼
  PySpark session
      │
      ▼
  Unity Catalog  ←── registers table metadata
      │
      ▼
    MinIO  ←── stores actual data files (Parquet/Delta)
```

## Step 0: Check environment variables
These are injected by JupyterHub (configured in Helm values)

In [ ]:
import os

print("MINIO_ENDPOINT:", os.environ.get("MINIO_ENDPOINT"))
print("MINIO_ACCESS_KEY:", os.environ.get("MINIO_ACCESS_KEY"))
print("UNITY_CATALOG_ENDPOINT:", os.environ.get("UNITY_CATALOG_ENDPOINT"))

## Step 1: Create MinIO 'warehouse' bucket
Unity Catalog needs this bucket to store table data.

In [ ]:
!pip install minio --quiet

In [ ]:
from minio import Minio

minio_client = Minio(
    "minio.minio.svc.cluster.local:9000",
    access_key=os.environ["MINIO_ACCESS_KEY"],
    secret_key=os.environ["MINIO_SECRET_KEY"],
    secure=False,
)

bucket_name = "warehouse"
if not minio_client.bucket_exists(bucket_name):
    minio_client.make_bucket(bucket_name)
    print(f"Created bucket: {bucket_name}")
else:
    print(f"Bucket already exists: {bucket_name}")

print("All buckets:", [b.name for b in minio_client.list_buckets()])

## Step 2: Create Unity Catalog catalog + schema via REST API
The Spark connector works with existing catalogs/schemas, so we create them first.

In [ ]:
import requests
import json

UC_ENDPOINT = os.environ["UNITY_CATALOG_ENDPOINT"]

# Check UC is reachable
resp = requests.get(f"{UC_ENDPOINT}/api/2.1/unity-catalog/catalogs")
print("UC catalogs response:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

In [ ]:
CATALOG_NAME = "tuantm"
SCHEMA_NAME = "demo"

# Create catalog
resp = requests.post(
    f"{UC_ENDPOINT}/api/2.1/unity-catalog/catalogs",
    json={"name": CATALOG_NAME, "comment": "Demo catalog for tuantm"},
)
if resp.status_code == 200:
    print(f"Created catalog: {CATALOG_NAME}")
elif resp.status_code == 409:
    print(f"Catalog already exists: {CATALOG_NAME}")
else:
    print(f"Create catalog response: {resp.status_code} - {resp.text}")

# Create schema
resp = requests.post(
    f"{UC_ENDPOINT}/api/2.1/unity-catalog/schemas",
    json={
        "name": SCHEMA_NAME,
        "catalog_name": CATALOG_NAME,
        "comment": "Demo schema",
    },
)
if resp.status_code == 200:
    print(f"Created schema: {CATALOG_NAME}.{SCHEMA_NAME}")
elif resp.status_code == 409:
    print(f"Schema already exists: {CATALOG_NAME}.{SCHEMA_NAME}")
else:
    print(f"Create schema response: {resp.status_code} - {resp.text}")

## Step 3: Create Spark session with Unity Catalog connector

This downloads required JARs from Maven Central:
- `unitycatalog-spark` — UC Spark connector
- `delta-spark` — Delta Lake format support
- `hadoop-aws` + `aws-java-sdk-bundle` — S3A filesystem for MinIO

**Note:** First run takes ~2 minutes to download JARs.

In [ ]:
from pyspark.sql import SparkSession

# Stop existing Spark session if any
try:
    spark.stop()
except:
    pass

UC_URI = os.environ["UNITY_CATALOG_ENDPOINT"]
MINIO_ENDPOINT = os.environ["MINIO_ENDPOINT"]
MINIO_ACCESS_KEY = os.environ["MINIO_ACCESS_KEY"]
MINIO_SECRET_KEY = os.environ["MINIO_SECRET_KEY"]

# Image: quay.io/jupyter/pyspark-notebook:latest
# Spark 4.1.1, Scala 2.13.17, OpenJDK 17

spark = (
    SparkSession.builder
    .appName("UC-MinIO-Demo")
    # --- Maven packages (downloaded on first run) ---
    .config(
        "spark.jars.packages",
        ",".join([
            "io.unitycatalog:unitycatalog-spark_2.13:0.4.0",
            "io.delta:delta-spark_2.13:4.1.0",
            "org.apache.hadoop:hadoop-aws:3.4.2",
        ])
    )
    # --- Delta Lake extensions ---
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    # --- Unity Catalog as the default catalog ---
    .config("spark.sql.catalog.unity", "io.unitycatalog.spark.UCSingleCatalog")
    .config("spark.sql.catalog.unity.uri", UC_URI)
    .config("spark.sql.catalog.unity.token", "")
    .config("spark.sql.defaultCatalog", "unity")
    # --- S3A / MinIO configuration ---
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    # --- Performance tuning for local ---
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark session created successfully!")

## Step 4: Create a table via Unity Catalog
The table metadata is stored in UC, data files are stored in MinIO.

In [ ]:
# List available catalogs
spark.sql("SHOW CATALOGS").show()

In [ ]:
# Use our catalog and schema
spark.sql("USE CATALOG tuantm")
spark.sql("USE SCHEMA demo")

# Show existing schemas
spark.sql("SHOW SCHEMAS").show()

In [ ]:
# Create a Delta table — data will be stored in MinIO at s3a://warehouse/tuantm/demo/users
spark.sql("""
    CREATE TABLE IF NOT EXISTS tuantm.demo.users (
        id INT,
        name STRING,
        email STRING,
        created_at TIMESTAMP
    )
    USING delta
    LOCATION 's3a://warehouse/tuantm/demo/users'
""")

print("Table created: tuantm.demo.users")

## Step 5: Insert data

In [ ]:
spark.sql("""
    INSERT INTO tuantm.demo.users VALUES
    (1, 'Tuan TM', 'tuantm@example.com', current_timestamp()),
    (2, 'Alice', 'alice@example.com', current_timestamp()),
    (3, 'Bob', 'bob@example.com', current_timestamp())
""")

print("Inserted 3 rows")

## Step 6: Query data

In [ ]:
# Simple SELECT
spark.sql("SELECT * FROM tuantm.demo.users").show()

In [ ]:
# Use DataFrame API
df = spark.table("tuantm.demo.users")
print(f"Row count: {df.count()}")
df.printSchema()

In [ ]:
# Filter with SQL
spark.sql("""
    SELECT name, email 
    FROM tuantm.demo.users 
    WHERE name LIKE '%Tuan%'
""").show()

## Step 7: Verify data in MinIO
Check that Delta files were actually written to MinIO.

In [ ]:
from minio import Minio

minio_client = Minio(
    "minio.minio.svc.cluster.local:9000",
    access_key=os.environ["MINIO_ACCESS_KEY"],
    secret_key=os.environ["MINIO_SECRET_KEY"],
    secure=False,
)

print("Files in s3://warehouse/tuantm/demo/users/:")
for obj in minio_client.list_objects("warehouse", prefix="tuantm/demo/users/", recursive=True):
    print(f"  {obj.object_name} ({obj.size} bytes)")

## Step 8: Verify table metadata in Unity Catalog
Check that UC has the table registered.

In [ ]:
resp = requests.get(f"{UC_ENDPOINT}/api/2.1/unity-catalog/tables", params={"catalog_name": "tuantm", "schema_name": "demo"})
print(json.dumps(resp.json(), indent=2))

## Step 9: Create another table — demo a simple ETL
Show that you can create DataFrames and register them as UC tables.

In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col, upper, current_timestamp

# Create sample orders data
orders_data = [
    Row(order_id=101, user_id=1, product="Laptop", amount=1200.0),
    Row(order_id=102, user_id=2, product="Mouse", amount=25.0),
    Row(order_id=103, user_id=1, product="Keyboard", amount=75.0),
    Row(order_id=104, user_id=3, product="Monitor", amount=450.0),
    Row(order_id=105, user_id=2, product="Headphones", amount=150.0),
]

orders_df = spark.createDataFrame(orders_data)

# Write as Delta table registered in Unity Catalog
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", "s3a://warehouse/tuantm/demo/orders") \
    .saveAsTable("tuantm.demo.orders")

print("Created table: tuantm.demo.orders")

In [ ]:
# Join users with orders
spark.sql("""
    SELECT u.name, o.product, o.amount
    FROM tuantm.demo.users u
    JOIN tuantm.demo.orders o ON u.id = o.user_id
    ORDER BY o.amount DESC
""").show()

In [ ]:
# Aggregate: total spending per user
spark.sql("""
    SELECT u.name, COUNT(*) as num_orders, SUM(o.amount) as total_spent
    FROM tuantm.demo.users u
    JOIN tuantm.demo.orders o ON u.id = o.user_id
    GROUP BY u.name
    ORDER BY total_spent DESC
""").show()

In [ ]:
# List all tables in our schema
spark.sql("SHOW TABLES IN tuantm.demo").show()

## Summary

What we demonstrated:
- **Unity Catalog** stores table metadata (catalog → schema → table)
- **MinIO** stores the actual data files (Delta/Parquet format)
- **PySpark** with UC Spark connector can create, read, write, and join tables
- Tables are accessible via both SQL and DataFrame API

### Architecture
```
┌────────────────┐
│  JupyterHub    │
│  (this notebook)│
└───────┬────────┘
        │ PySpark + UC Spark Connector
        ▼
┌────────────────┐     ┌────────────────┐
│ Unity Catalog  │     │     MinIO       │
│  (metadata)    │     │  (data files)   │
│                │     │                 │
│ tuantm.demo    │     │ s3://warehouse/ │
│  ├── users     │────▶│  tuantm/demo/   │
│  └── orders    │     │   ├── users/    │
│                │     │   └── orders/   │
└────────────────┘     └────────────────┘
```